In [3]:
import json, sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm


## Loading the environment

In [4]:
DB = Path("../data/hg19.db")
MUT = Path("../data/simple_breast.csv")

Mutations

In [12]:
mutations = pd.read_csv(MUT,
                        dtype={"sampleID":str,
                               "chr":str,
                               "pos":int,
                               "ref":str,
                               "mut":str}).iloc[:, :5]

In [13]:
mutations.head()

,sampleID,chr,pos,ref,mut
0,Sample_1,1,871244,G,C
1,Sample_1,1,6648841,C,G
2,Sample_1,1,17557072,G,A
3,Sample_1,1,22838492,G,C
4,Sample_1,1,27097733,G,A


In [9]:
mutations.shape

(18637, 5)

In [10]:
mutations.dropna().reset_index(drop=True, inplace=True)

In [11]:
mutations.shape

(18637, 5)

Exclude 0/0 mutations

In [14]:
mutations = mutations[mutations["ref"] != mutations["mut"]].reset_index(drop=True)

In [15]:
mutations.head()

,sampleID,chr,pos,ref,mut
0,Sample_1,1,871244,G,C
1,Sample_1,1,6648841,C,G
2,Sample_1,1,17557072,G,A
3,Sample_1,1,22838492,G,C
4,Sample_1,1,27097733,G,A


In [16]:
mutations.shape

(18637, 5)

Contiguous / Duplicate mutations

In [ ]:
mutations[mutations.sort_values(
        ["sampleID", "chr", "pos"])
        .groupby("sampleID")["pos"].diff() == 1]

,sampleID,chr,pos,ref,mut


In [28]:
mutations[mutations[["chr", "pos", "ref", "mut"]].duplicated()].sort_values(["pos", "ref", "mut", "chr"])

,sampleID,chr,pos,ref,mut
13011,Sample_5,11,533874,T,C
7328,Sample_174,X,5811421,C,T
18157,Sample_91,17,7577022,G,A
16941,Sample_8,17,7577094,G,A
8685,Sample_187,17,7577539,G,A
...,...,...,...,...,...
14638,Sample_61,2,209113112,C,T
16893,Sample_79,2,209113112,C,T
18204,Sample_92,2,209113112,C,T
18579,Sample_98,2,209113112,C,T


## Load Reference Coding sequence

In [29]:
conn = sqlite3.connect(DB)

In [30]:
refcds = pd.read_sql("SELECT * FROM refcds", conn)

In [32]:
refcds.head()

,gene_name,gene_id,protein_id,CDS_length,chr,strand,intervals_splice,seq_cds,seq_cds1up,seq_cds1down,seq_splice,seq_splice1up,seq_splice1down,L
0,A1BG,ENSG00000121410,ENSP00000263100,1488,19,-1,"[58858396,58858397,58858714,58858717,58858718,...",ATGTCCATGCTCGTGGTCTTTCTCTTGCTGTGGGGTGTCACCTGGG...,CATGTCCATGCTCGTGGTCTTTCTCTTGCTGTGGGGTGTCACCTGG...,TGTCCATGCTCGTGGTCTTTCTCTTGCTGTGGGGTGTCACCTGGGG...,GACTGGAGTGGAGTGGACTGGAGTGGAGTGGAGTG,ATAGGACAGGACAGGACAGGACAGAACAGTACAGG,AGGGTGGCGTAGCGTCGTGTAGCGTTGTGTGGGGT,"[[0,7,0,0],[3,4,0,0],[0,7,0,0],[0,12,0,0],[1,1..."
1,A1CF,ENSG00000148584,ENSP00000363105,1785,10,-1,"[52566641,52566642,52569649,52569652,52569653,...",ATGGAATCAAATCACAAATCCGGGGATGGATTGAGCGGCACTCAGA...,AATGGAATCAAATCACAAATCCGGGGATGGATTGAGCGGCACTCAG...,TGGAATCAAATCACAAATCCGGGGATGGATTGAGCGGCACTCAGAA...,GAGTGGAGTGGAGTGGACTGGAGTGGAGTGGAGTGGAGTGGAGTGG...,ACTGGACCGTATGGGATAGGACAGGACGGGACAGGACAGGACAGAA...,GGGATCGTGTAGTATAGAATGGGATGGAATGGCTTAGCATAGTGTG...,"[[0,66,0,0],[22,44,0,0],[0,55,11,0],[0,24,0,0]..."
2,A2M,ENSG00000175899,ENSP00000323929,4425,12,-1,"[9220436,9220437,9220774,9220777,9220778,92208...",ATGGGGAAGAACAAACTCCTTCATCCAAGTCTGGTTCTTCTCCTCT...,CATGGGGAAGAACAAACTCCTTCATCCAAGTCTGGTTCTTCTCCTC...,TGGGGAAGAACAAACTCCTTCATCCAAGTCTGGTTCTTCTCCTCTT...,GAGTGGAGTGGAGTGGAGTGGAGTGGAGTGGAGTGGAGTGGAGTGG...,ACAGGACAGGACAGGACGGGACAGGATAGGACGGGAAAGGACGGTA...,AGCATAGTGTGGAATCGTATTGAATAGAGTGGTTTAGAGTCGTATG...,"[[0,89,0,0],[21,68,0,0],[0,74,15,0],[0,63,0,2]..."
3,A2ML1,ENSG00000166535,ENSP00000299698,4365,12,1,"[8975310,8975311,8975314,8975776,8975777,89759...",ATGTGGGCTCAGCTCCTTCTAGGAATGTTGGCCCTATCACCAGCCA...,GATGTGGGCTCAGCTCCTTCTAGGAATGTTGGCCCTATCACCAGCC...,TGTGGGCTCAGCTCCTTCTAGGAATGTTGGCCCTATCACCAGCCAT...,GTGAGGTGAGGTGAGGTGAGGTGAGGTGAGGTGAGGTGAGGTGAGG...,CGATATGACAGGACAGGACAGGACAGGGCAGGATAGGATAGGACAG...,TGTGATACGGTAAGTTGTGTTACGGTATGTTAAGGTGTGATATGGT...,"[[0,75,0,1],[17,58,0,1],[0,60,15,1],[0,60,0,1]..."
4,A3GALT2,ENSG00000184389,ENSP00000475261,1023,1,-1,"[33773055,33773056,33777648,33777651,33777652,...",ATGGCTCTCAAGGAGGGACTCAGGGCCTGGAAGAGAATCTTCTGGC...,TATGGCTCTCAAGGAGGGACTCAGGGCCTGGAAGAGAATCTTCTGG...,TGGCTCTCAAGGAGGGACTCAGGGCCTGGAAGAGAATCTTCTGGCG...,GAGTGGAGTGGAATGGAGTG,ATAGGACAGGACAGGACAGG,AGGATGGAATGGAATGGGGT,"[[0,1,0,1],[0,1,0,1],[0,1,0,1],[0,6,0,0],[1,5,..."


In [44]:
# intervals = pd.read_sql("SELECT gene_name, start, end FROM cds_intervals ORDER BY start", conn)

# intervals = pd.read_sql("SELECT gene_name, start, end FROM cds_intervals ORDER BY gene_name, rowid", conn)

# Sort by order of exonic regions
intervals = pd.read_sql("SELECT gene_name, start, end FROM cds_intervals ORDER BY gene_name, start", conn)

In [45]:
intervals.head()

,gene_name,start,end
0,A1BG,58858388,58858395
1,A1BG,58858719,58859006
2,A1BG,58861736,58862017
3,A1BG,58862757,58863053
4,A1BG,58863649,58863921


In [46]:
gene_cds = {name: list(zip(df["start"], df["end"])) for name, df in intervals.groupby("gene_name")}

In [50]:
all(type(row["CDS_length"]) == int for _, row in refcds[:2].iterrows())

True

In [51]:
all(type(row["strand"]) == int for _, row in refcds[:2].iterrows())

True

In [65]:
json.loads(refcds.iloc[0]["intervals_splice"])[:10]

[58858396,
 58858397,
 58858714,
 58858717,
 58858718,
 58859007,
 58859008,
 58861731,
 58861734,
 58861735]

In [71]:
RefCDS = {
    row["gene_name"]: {
        "gene_name": row["gene_name"],
        "gene_id": row["gene_id"],
        "protein_id": row["protein_id"],
        "CDS_length": int(row["CDS_length"]),
        "chr": str(row["chr"]),
        "strand": int(row["strand"]),
        "intervals_cds": gene_cds.get(row["gene_name"], []),
        "intervals_splice": set(json.loads(row["intervals_splice"])),
        "seq_cds": list(row["seq_cds"] or ""),
        "seq_cds1up": list(row["seq_cds1up"] or ""),
        "seq_cds1down": list(row["seq_cds1down"] or ""),
        "seq_splice": list(row["seq_splice"] or ""),
        "seq_splice1up": list(row["seq_splice1up"] or ""),
        "seq_splice1down": list(row["seq_splice1down"] or ""),
        "L": np.array(json.loads(row["L"]), dtype=np.int32),
        "N": np.zeros((192, 4), dtype=np.int32),
    }
    for _, row in refcds.iterrows()
}

In [74]:
RefCDS["BRCA1"].keys()

dict_keys(['gene_name', 'gene_id', 'protein_id', 'CDS_length', 'chr', 'strand', 'intervals_cds', 'intervals_splice', 'seq_cds', 'seq_cds1up', 'seq_cds1down', 'seq_splice', 'seq_splice1up', 'seq_splice1down', 'L', 'N'])

In [75]:
genes = list(RefCDS.keys())
gidx = {g: i for i, g in enumerate(genes)}

In [76]:
genes[0], gidx[genes[0]]

('A1BG', 0)